# ResolveX — 02: Validate + Package Submission (run LOCALLY)

Run this **after** copying the files listed at the end of the Kaggle notebook
into this project's `output/` and `experiments/` folders. This step needs your
local filesystem (to build the final zip) so it isn't a Kaggle step.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))


## Run the official validator against your downloaded Kaggle outputs

In [ ]:
import subprocess

result = subprocess.run(
    [
        sys.executable, str(REPO_ROOT / "resources" / "utils" / "validate_submission.py"),
        "--matching", str(REPO_ROOT / "output" / "matching_results.tsv"),
        "--candidate", str(REPO_ROOT / "output" / "candidate_pairs.tsv"),
        "--test-dir", str(REPO_ROOT / "resources" / "dataset" / "test"),
    ],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Validator failed — fix the issues above before packaging."


## Package the final submission zip

In [ ]:
import zipfile

from resolvex.config import CANDIDATE_PAIRS_PATH, MATCHING_RESULTS_PATH, ROOT

SUBMISSION_ZIP = ROOT / "resolvex_submission.zip"
doc_path = ROOT / "docs" / "Documentation_template.md"
readme_path = ROOT / "README.md"
requirements_path = ROOT / "requirements.txt"
src_dir = ROOT / "src"

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(MATCHING_RESULTS_PATH, "output/matching_results.tsv")
    zf.write(CANDIDATE_PAIRS_PATH, "output/candidate_pairs.tsv")
    for path in src_dir.rglob("*"):
        if path.is_file() and "__pycache__" not in path.parts:
            zf.write(path, f"code/business_entity_resolution/{path.relative_to(ROOT).as_posix()}")
    if readme_path.exists():
        zf.write(readme_path, "code/business_entity_resolution/README.md")
    if requirements_path.exists():
        zf.write(requirements_path, "code/business_entity_resolution/requirements.txt")
    if doc_path.exists():
        zf.write(doc_path, "Documentation_template.md")

print(f"Wrote {SUBMISSION_ZIP}")
